In [ ]:
# Deploy Manny: log the agent with MLflow → register in Unity Catalog → serve with agents.deploy
# agents.deploy creates the Model Serving endpoint with MLflow tracing, an inference table (request/response
# payloads in UC Delta) and the review app. Lakebase and Geoapify credentials reach the endpoint as
# environment variables backed by the `maintops` secret scope — never in code.

In [ ]:
%pip install -q -U "mlflow>=3.1" databricks-agents databricks-sdk openai "psycopg[binary]>=3.1" psycopg_pool requests

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))   # repo root: manny.py imports maintops_core
import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksVectorSearchIndex

UC_MODEL      = "bootcamp_students.maintops.manny"
ENDPOINT_NAME = "maintops-manny"
LLM_ENDPOINT  = "databricks-claude-sonnet-4-6"
FAQ_INDEX     = "bootcamp_students.maintops.faq_index"

w = WorkspaceClient()
me = w.current_user.me().user_name
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{me}/maintops_manny")

# Needed while logging (the input example is run once); the endpoint gets them from the secret scope
os.environ["LAKEBASE_PG_URL"]  = dbutils.secrets.get("maintops", "lakebase_pg_url")
os.environ["GEOAPIFY_API_KEY"] = dbutils.secrets.get("maintops", "geoapify_api_key")

In [ ]:
input_example = {
    "input": [{"role": "user", "content": "How does MaintOps choose a handyman?"}],
    "custom_inputs": {"role": "visitor"},
}
with mlflow.start_run(run_name="manny"):
    info = mlflow.pyfunc.log_model(
        name="manny",
        python_model="manny.py",                      # models-from-code: agent/manny.py
        code_paths=["../maintops_core"],              # shared services packaged with the model
        resources=[DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
                   DatabricksVectorSearchIndex(index_name=FAQ_INDEX)],
        pip_requirements=["mlflow>=3.1", "databricks-sdk", "openai", "httpx", "psycopg[binary]>=3.1",
                          "psycopg_pool", "requests"],
        input_example=input_example,
    )
registered = mlflow.register_model(info.model_uri, UC_MODEL)
print(f"Registered {UC_MODEL} version {registered.version}")

In [ ]:
from databricks import agents

deployment = agents.deploy(
    UC_MODEL, registered.version,
    endpoint_name=ENDPOINT_NAME,
    scale_to_zero=False,                              # the web app needs Manny without cold starts
    environment_vars={
        "LAKEBASE_PG_URL":  "{{secrets/maintops/lakebase_pg_url}}",
        "GEOAPIFY_API_KEY": "{{secrets/maintops/geoapify_api_key}}",
        "MANNY_LLM_ENDPOINT": LLM_ENDPOINT,
    },
    tags={"project": "maintops"},
)
print(deployment)

In [ ]:
# Wait until the new version serves, then smoke-test it
deadline = time.time() + 45 * 60
while time.time() < deadline:
    ep = w.serving_endpoints.get(ENDPOINT_NAME)
    ready, updating = str(ep.state.ready), str(ep.state.config_update)
    print(f"{time.strftime('%H:%M:%S')} ready={ready} config_update={updating}")
    if ready.endswith("READY") and not updating.endswith("IN_PROGRESS"):
        break
    time.sleep(60)
else:
    raise TimeoutError("endpoint not ready after 45 minutes")

resp = w.api_client.do("POST", f"/serving-endpoints/{ENDPOINT_NAME}/invocations", body={
    "input": [{"role": "user", "content": "Can I choose the handyman myself?"}],
    "custom_inputs": {"role": "visitor"}})
print(resp["output"][0]["content"][0]["text"])